<!-- copybara:strip_begin -->
> [!IMPORTANT]
> **Open-Source Codelab Notice**
> This notebook is intended for usage in a public environment (like public Google Colab) and demonstrates the open-source version of DPSynth. It installs dependencies from public repositories and uses installation methods that are not supported inside Google3.
>
> Google employees looking for internal solutions or documentation should refer to **go/dpsynth** (or the relevant internal documentation).
<!-- copybara:strip_end -->

# DPSynth Quickstart Demo

This notebook demonstrates how to use DPSynth to generate Differentially Private synthetic tabular data using the In-Memory DataFrame API.

In [ ]:
# Install dependencies
!pip install git+https://github.com/google/dpsynth.git

In [ ]:
import dpsynth
from dpsynth import discrete_mechanisms
from dpsynth import domain
import pandas as pd

# 1. Create a simple dummy dataset
sensitive_df = pd.DataFrame({
    'age': [25, 30, 25, 35, 40, 45, 50, 25, 30, 35],
    'city': ['NY', 'London', 'London', 'Paris', 'NY', 'Paris', 'London', 'NY', 'Paris', 'London']
})

print("Original Sensitive Data:")
print(sensitive_df)

# 2. Define domain schema
attribute_domains = {
    'age': domain.NumericalAttribute(min_value=20, max_value=60),
    'city': domain.CategoricalAttribute(possible_values=['NY', 'London', 'Paris'])
}

# 3. Configure the synthesis mechanism (MST as default)
mst_config = discrete_mechanisms.MSTConfig(seed=42)

# 4. Generate Differentially Private synthetic data
synthetic_df = dpsynth.generate(
    data=sensitive_df,
    domains=attribute_domains,
    epsilon=1.0,
    delta=1e-5,
    discrete_config=mst_config,
)

print("\nGenerated Synthetic Data:")
print(synthetic_df)

# Scalable Beam API Example

This example demonstrates how to use the Scalable Beam API to generate synthetic data in a distributed manner using Apache Beam.

In [ ]:
import apache_beam as beam
from dpsynth import data_generation
from dpsynth.pipeline_transformations import types
from dpsynth import domain
from dpsynth.dataset_descriptors import csv_descriptor
import pipeline_dp
import pandas as pd

# 1. Create a simple dummy dataset in-memory for demonstration
sensitive_data = [
    {'age': 25, 'city': 'NY'},
    {'age': 30, 'city': 'London'},
    {'age': 25, 'city': 'London'},
    {'age': 35, 'city': 'Paris'},
    {'age': 40, 'city': 'NY'}
]

# 2. Define domain schema
attribute_domains = {
    'age': domain.NumericalAttribute(min_value=20, max_value=60),
    'city': domain.CategoricalAttribute(possible_values=['NY', 'London', 'Paris'])
}

# 3. Create descriptor
# We use a dummy dataframe to create the descriptor
sample_df = pd.DataFrame(sensitive_data)
descriptor = csv_descriptor.get_dataset_descriptor_for_csv(
    dataframe=sample_df,
    field_names=["age", "city"]
)
descriptor.update_from_domain_specification(attribute_domains)

# 4. Configure the synthesis mechanism
config = data_generation.DataGenerationConfig(
    epsilon=1.0,
    delta=1e-5,
    mechanism=data_generation.Mechanism.MST,
    dataset_descriptor=descriptor,
    data_format=types.DataFormat.CSV,
    num_out_records=10,
)

# 5. Run the pipeline with BeamBackend
with beam.Pipeline() as pipeline:
    # Load data into PCollection
    raw_records = pipeline | "CreateData" >> beam.Create(sensitive_data)
    
    beam_backend = pipeline_dp.BeamBackend()
    
    synthetic_records = data_generation.generate(
        input_data=raw_records,
        config=config,
        backend=beam_backend,
    )
    
    # Print results
    synthetic_records | "Print" >> beam.Map(print)